In [4]:
import json
import cobra

In [5]:
# Load the model
model = cobra.io.read_sbml_model("model.xml")

In [2]:
# Load the json file
with open('simulations/pFBA/results/mbm_Glucose _pfba_fluxes.json', 'r') as file:
    fluxes = json.load(file)

In [7]:
# Define the metabolite object for NADPH
nadph = model.metabolites.get_by_id("cpd00005_c0") 

In [10]:
model.reactions.rxn05338_c0

Reaction identifier,rxn05338_c0
Name,(3R)-3-Hydroxydecanoyl-[acyl-carrier-protein]:NADP+ oxidoreductase [c0]
Memory address,0x11bc86010
Stoichiometry,cpd00006_c0 + cpd11482_c0 <=> cpd00005_c0 + cpd00067_c0 + cpd11487_c0 NADP + (3R)-3-Hydroxydecanoyl-ACP <=> NADPH + H+ + 3-Oxodecanoyl-ACP
GPR,WP_014976326.1
Lower bound,-1000.0
Upper bound,1000.0


In [14]:
# Get reactions that produce NADPH
print ("| Reaction ID | Reaction Name | Reaction | Flux | NADPH Production Rate |")
print("| --- | --- | --- | --- | --- |")
for rxn in nadph.reactions:
    flux = fluxes.get(rxn.id, 0)
    if abs(flux) > 1e-6:
        coeff = rxn.get_coefficient(nadph)
        if (coeff > 0 and flux > 0) or (coeff < 0 and flux < 0):  # Only consider reactions that produce NADPH
            print(f"| {rxn.id} | {rxn.name} | {rxn.build_reaction_string(use_metabolite_names=True)} | {flux:.3f} | {coeff*flux:.3f} |")

| Reaction ID | Reaction Name | Reaction | Flux | NADPH Production Rate |
| --- | --- | --- | --- | --- |
| rxn00085_c0 | L-Glutamate:NADP+ oxidoreductase (transaminating) [c0] | NADP + 2.0 L-Glutamate <=> NADPH + 2-Oxoglutarate + L-Glutamine + H+ | 1.304 | 1.304 |
| rxn01387_c0 | Isocitrate:NADP+ oxidoreductase [c0] | NADP + Isocitrate <=> NADPH + H+ + Oxalosuccinate | 4.577 | 4.577 |
| rxn00907_c0 | 5,10-methylenetetrahydrofolate:NADP+ oxidoreductase [c0] | NADP + 5-10-Methylenetetrahydrofolate <=> NADPH + 5-10-Methenyltetrahydrofolate | 0.639 | 0.639 |


In [ ]:
# Correct rxn00085
rxn = model.reactions.get_by_id("rxn00085_c0")
# Update the model with the new reaction definition
rxn.add_metabolites({met: -coeff for met, coeff in rxn.metabolites.items()}, combine=False)
# Make it irreversible in the forward direction
rxn.lower_bound = 0


In [20]:
rxn

Reaction identifier,rxn00085_c0
Name,L-Glutamate:NADP+ oxidoreductase (transaminating) [c0]
Memory address,0x11c04f250
Stoichiometry,cpd00005_c0 + cpd00024_c0 + cpd00053_c0 + cpd00067_c0 --> cpd00006_c0 + 2.0 cpd00023_c0 NADPH + 2-Oxoglutarate + L-Glutamine + H+ --> NADP + 2.0 L-Glutamate
GPR,WP_012519580.1 and WP_014950543.1
Lower bound,0
Upper bound,1000.0


In [21]:
# Save the modified model
cobra.io.write_sbml_model(model, "model.xml")